<h2 style="color:#FF7A70;">Clase Proyecto Final: Global Economic Analysis Report 2025</h2>

<p><strong>Curso:</strong> Lenguaje y Programación II</p>

<p><strong>Integrantes del grupo:</strong></p>
<ul>
  <li>Castillo Flores, Hellary Mayte — <em>20240699</em></li>
  <li>Cruz Lozano, Gianella Alejandra — <em>20240705</em></li>
  <li>Tineo Balcázar, Daniela Rosa — <em>20231510</em></li>
</ul>

<h2 style="color:#FF7A70;">Objetivo:</h2>
<p>
Analizar el <strong>Top 10 de países por PIB Nominal</strong> utilizando datos oficiales
del <strong>Banco Mundial</strong>, e incorporar una métrica comparativa adicional llamada
<strong>Capacidad BTC</strong>, basada en el precio actual de Bitcoin, con el fin de
ofrecer una visión económica alternativa y moderna.
</p>

<h2 style="color:#FF7A70;">Fuentes de datos:</h2>
<ul>
  <li><strong>World Bank API</strong> – Indicadores macroeconómicos</li>
  <li><strong>CoinGecko API</strong> – Precio actual de Bitcoin (USD)</li>
</ul>
<h2 style="color:#FF7A70;">Parte 1: Carga de datos y APIs</h2>
<p>En esta sección vamos a:</p>
<ol>
  <li>Descargar el precio de Bitcoin desde CoinGecko.</li>
  <li>Descargar indicadores económicos (PIB, PIB per cápita, crecimiento) desde World Bank.</li>
  <li>Obtener la población de cada país usando REST Countries (optimizado para rapidez).</li>
  <li>Calcular la Capacidad BTC y PIB per cápita real.</li>
</ol>





In [1]:
import pandas as pd
import requests
import wbgapi as wb
import warnings

warnings.filterwarnings("ignore")  # Ignorar advertencias

# -----------------------------
# Transformar valores numéricos en cadenas de texto
# -----------------------------
def format_money(val):
    if pd.isna(val): return "N/A"
    if val >= 1e12: return f"{val/1e12:,.2f} Trillion USD"
    if val >= 1e9: return f"{val/1e9:,.2f} Billion USD"
    return f"{val:,.2f} USD"

def format_btc(val):
    if pd.isna(val): return "N/A"
    return f"{val:,.0f} BTC"

def format_percent(val):
    if pd.isna(val): return "N/A"
    return f"{val:.2f} %"

def format_population(val):
    if pd.isna(val): return "N/A"
    return f"{val:,}"

# -----------------------------
# Cargar datos
# -----------------------------
# Precio BTC
btc_price = requests.get(
    "https://api.coingecko.com/api/v3/simple/price",
    params={"ids": "bitcoin", "vs_currencies": "usd"}
).json()["bitcoin"]["usd"]

# Indicadores económicos
indicadores = {
    "NY.GDP.MKTP.CD": "PIB_Nominal",
    "NY.GDP.PCAP.CD": "PIB_Per_Capita",
    "NY.GDP.MKTP.KD.ZG": "Crecimiento_PIB"
}

df = wb.data.DataFrame(indicadores.keys(), labels=True, mrnev=1).reset_index()
df = df.rename(columns={
    "Country": "Pais",
    "NY.GDP.MKTP.CD": "PIB_Nominal",
    "NY.GDP.PCAP.CD": "PIB_Per_Capita",
    "NY.GDP.MKTP.KD.ZG": "Crecimiento_PIB"
})

# Eliminamos agregados
paises_validos = [c["id"] for c in wb.economy.list() if not c["aggregate"]]
df = df[df["economy"].isin(paises_validos)]
df = df.dropna(subset=["PIB_Nominal"])

# -----------------------------
# Poblaciones (REST Countries)
# -----------------------------
resp = requests.get("https://restcountries.com/v3.1/all").json()
poblaciones_dict = {
    c["name"]["common"].lower(): c.get("population", None)
    for c in resp
    if isinstance(c, dict) and "name" in c and "common" in c["name"]
}

df["Poblacion"] = df["Pais"].str.lower().map(poblaciones_dict)
df["Poblacion"] = df["Poblacion"].fillna(df["PIB_Nominal"] / df["PIB_Per_Capita"])

# -----------------------------
# Calcular métricas
# -----------------------------
df["PIB_Per_Capita_Real"] = df["PIB_Nominal"] / df["Poblacion"]

df["Capacidad_BTC"] = df["PIB_Nominal"] / btc_price

df["Capacidad_BTC_per_capita"] = df["Capacidad_BTC"] / df["Poblacion"]

# -----------------------------
# Orden final
# -----------------------------
df = df.sort_values("PIB_Nominal", ascending=False).reset_index(drop=True)
df.index += 1






<h2 style="color:#FF7A70;">Parte 2: Menú interactivo y consultas</h2>
<p>Esta sección:</p> 

<ol>
<li>Mostrará el menú de opciones en Jupyter.</li>
<li>Permitirá consultar Top 10 PIB o Capacidad BTC.</li>
<li>Consultará información (completa) de un país específico, al cual accederemos mediante un imput.</li>
</ol>

In [ ]:
# -----------------------------
# Funciones del menú
# -----------------------------
def mostrar_top10(df, columna, titulo):
    print(f"\n🏆 {titulo}")
    print("-" * 50)
    for i, row in df.head(10).iterrows():
        if columna in ["PIB_Nominal", "PIB_Per_Capita", "PIB_Per_Capita_Real"]:
            valor = format_money(row[columna])
        elif columna == "Capacidad_BTC":
            valor = format_btc(row[columna])
        elif columna == "Crecimiento_PIB":
            valor = format_percent(row[columna])
        elif columna == "Poblacion":
            valor = format_population(row[columna])
        else:
            valor = row[columna]
        print(f"{i}. {row['Pais']:<20} {valor}")

def consultar_pais(df):
    pais = input("\nIngrese el país (en inglés): ").strip()
    fila = df[df["Pais"].str.lower() == pais.lower()]
    if fila.empty:
        print("❌ País no encontrado.")
        return None

    row = fila.iloc[0]
    print("\n📊 INFORMACIÓN DEL PAÍS")
    print("-" * 40)
    print(f"País: {row['Pais']}")
    print(f"PIB Nominal: {format_money(row['PIB_Nominal'])}")
    print(f"PIB per Cápita (WB): {format_money(row['PIB_Per_Capita'])}")
    print(f"PIB per Cápita (Real): {format_money(row['PIB_Per_Capita_Real'])}")
    print(f"Crecimiento PIB: {format_percent(row['Crecimiento_PIB'])}")
    print(f"Población: {format_population(row['Poblacion'])}")
    print(f"Capacidad BTC: {format_btc(row['Capacidad_BTC'])}")
    return row

# -----------------------------
# Menú interactivo en Jupyter
# -----------------------------
consultas = []

while True:
    print("\n📌 MENÚ PRINCIPAL")
    print("1. Top 10 PIB")
    print("2. Top 10 Capacidad BTC")
    print("3. Consultar país")
    print("4. Salir")
    
    op = input("Seleccione opción: ")
    
    if op == "1":
        mostrar_top10(df, "PIB_Nominal", "TOP 10 PIB")
    elif op == "2":
        mostrar_top10(df.sort_values("Capacidad_BTC", ascending=False), "Capacidad_BTC", "TOP 10 CAPACIDAD BTC")
    elif op == "3":
        r = consultar_pais(df)
        if r is not None:
            consultas.append(r)
    elif op == "4":
        print("✅ Finalizando consultas...")
        break
    else:
        print("❌ Opción inválida")



📌 MENÚ PRINCIPAL
1. Top 10 PIB
2. Top 10 Capacidad BTC
3. Consultar país
4. Salir


<h2 style="color:#FF7A70;">Parte 3: Exportación HTML</h2>
<p>En esta sección vamos a:</p> 

<ol>
<li>Formatear los números .</li>
<li>Generar un HTML final acumulativo.</li>
</ol>


# -----------------------------
# Formatear DataFrame para HTML
# -----------------------------
def formatear_df_html(df):
    df_formateado = df.copy()
    for col in df_formateado.columns:
        if col in ["PIB_Nominal", "PIB_Per_Capita", "PIB_Per_Capita_Real"]:
            df_formateado[col] = df_formateado[col].apply(format_money)
        elif col in ["Capacidad_BTC", "Capacidad_BTC_per_capita"]:
            df_formateado[col] = df_formateado[col].apply(format_btc)
        elif col == "Crecimiento_PIB":
            df_formateado[col] = df_formateado[col].apply(format_percent)
        elif col == "Poblacion":
            df_formateado[col] = df_formateado[col].apply(format_population)
    return df_formateado

# -----------------------------
# Exportar HTML
# -----------------------------
top_pib = df.sort_values("PIB_Nominal", ascending=False).head(10)
top_btc_pc = df.sort_values("Capacidad_BTC_per_capita", ascending=False).head(10)
consultas_df = pd.DataFrame(consultas)

top_pib_html = formatear_df_html(top_pib)
top_btc_html = formatear_df_html(top_btc_pc)
consultas_html = formatear_df_html(consultas_df) if not consultas_df.empty else pd.DataFrame()

html = f"""
<html>
<head>
<meta charset="utf-8">
<title>Global Economic Report 2025</title>
<style>
body {{ font-family: Segoe UI; background:#f4f6f7; margin:40px; }}
h1 {{ color:#2c3e50; }}
h2 {{ color:#2980b9; border-bottom:2px solid #2980b9; }}
table {{ width:100%; border-collapse:collapse; margin-top:20px; }}
th {{ background:#2c3e50; color:white; padding:10px; }}
td {{ padding:8px; text-align:center; border-bottom:1px solid #ddd; }}
</style>
</head>
<body>

<h1>🌍 Global Economic Analysis Report 2025</h1>
<p><b>Bitcoin reference price:</b> ${btc_price:,} USD</p>

<h2>🏆 Top 10 Countries by GDP</h2>
{top_pib_html.to_html(index=True)}

<h2>🪙 Top 10 Countries by BTC per capita</h2>
{top_btc_html.to_html(index=True)}

<h2>🌎 Country Queries</h2>
{consultas_html.to_html(index=False) if not consultas_html.empty else "<p>No queries yet.</p>"}

</body>
</html>
"""

with open("REPORTE_ECONOMICO_GLOBAL_2025.html", "w", encoding="utf-8") as f:
    f.write(html)

print("✅ Reporte generado: REPORTE_ECONOMICO_GLOBAL_2025.html")


<h2 style="color:#FF7A70;">Conclusiones</h2>
<p>Después de analizar los datos económicos globales y la capacidad de cada país de “comprar” Bitcoin, podemos destacar:</p>

<ol>
<li>Los países con mayor PIB nominal no necesariamente tienen la mayor capacidad en BTC, ya que el precio de Bitcoin y la población influyen significativamente.</li>
<li>El PIB per cápita real, calculado usando población de REST Countries, permite comparar de manera más justa la riqueza relativa de los países.</li>
<li>El análisis combinado de PIB, población y Capacidad BTC proporciona una perspectiva completa sobre la economía y el potencial de adopción de criptomonedas a nivel global.</li>
<li>Automatizar la descarga de datos mediante APIs reduce errores y permite actualizar rápidamente el reporte año tras año.</li>
<li>El formato numérico y la visualización en HTML hacen que el reporte sea mucho más legible y presentable, facilitando la interpretación de los datos por distintos públicos.</li>
</ol>

<p>En resumen, este proyecto integra información económica confiable con datos de criptomonedas, ofreciendo un análisis claro, actualizado y visualmente amigable.</p>
